# Skip-Gram word2vec

Pure NumPy implementation of word2vec using skip-gram with negative sampling.

---

## 1. Setup and Imports

In [159]:
import numpy as np
import matplotlib.pyplot as pyplot
from collections import Counter

# Seed for reporducibility
np.random.seed(42)

## 2. Mini Corpus for Testing

Small dataset to verify our implementation before scaling to larger text.

In [160]:
# Mini corpus for testing
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are enemies",
    "the cat and the dog are friends",
    "the black cat chased the small mouse",
    "dogs love to play with their owners",
    "a lazy cat sleeps all day on the sofa"
]

print("Corpus:")
for i, sentence in enumerate(corpus):
    print(f" {i}: {sentence}")

Corpus:
 0: the cat sat on the mat
 1: the dog sat on the log
 2: cats and dogs are enemies
 3: the cat and the dog are friends
 4: the black cat chased the small mouse
 5: dogs love to play with their owners
 6: a lazy cat sleeps all day on the sofa


## 3. Tokenization

Convert sentences into lists of words and build flat word list.

In [161]:
# Tokenize (separates words)
def tokenize(text):
    return text.lower().split()


# Flatten all sentences in one list of words
all_words = []
for sentence in corpus:
    all_words.extend(tokenize(sentence))

print(f"Total words: {len(all_words)}")
print(f"Sample: {all_words[:10]}")

Total words: 47
Sample: ['the', 'cat', 'sat', 'on', 'the', 'mat', 'the', 'dog', 'sat', 'on']


## 4. Vocabulary Building

Create bidirectional mappings: word ↔ index.  
Sorted alphabetically for reproducibility.

In [162]:
# Build vocabulary
word_counts = Counter(all_words)
vocab = sorted(word_counts.keys())

# Create mappings
word_to_idx = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for word, idx in word_to_idx.items()} # Sorts alphabeticaly

vocab_size = len(vocab)

print(f"Vocabulary size: {vocab_size}")
print(f"\nWord to Index mapping (first 10):")
for word in list(vocab)[:10]:
    print(f" '{word}' -> {word_to_idx[word]}")

Vocabulary size: 29

Word to Index mapping (first 10):
 'a' -> 0
 'all' -> 1
 'and' -> 2
 'are' -> 3
 'black' -> 4
 'cat' -> 5
 'cats' -> 6
 'chased' -> 7
 'day' -> 8
 'dog' -> 9


## 5. Skip-Gram Training Pairs Generation

Generate (center_word, context_word) pairs using sliding window.  
Each pair represents one training example for the model.

In [163]:
def generate_training_data(corpus, word_to_idx, window_size = 2):
    """
    Generate (center_word_idx, context_word_idx) pairs for skip-gram.

    Args:
        corpus: List of senteces (strings)
        word_to_idx: Dictionary mapping word -> index
        window_size: How many words to look left/right

    Returns:
        pairs: List of tuples (center_idx, context_idx)
    """
    pairs = []

    for sentence in corpus:
        words = tokenize(sentence)
        word_indices = [word_to_idx[word] for word in words]

        for center_pos, center_idx in enumerate(word_indices):
            for offset in range(-window_size, window_size + 1):
                context_pos = center_pos + offset

                if offset == 0:
                    continue

                if context_pos < 0 or context_pos >= len(word_indices):
                    continue
                
                context_idx = word_indices[context_pos]
                pairs.append((center_idx, context_idx))

    return pairs

test_corpus = ["the cat sat on the mat"]
test_pairs = generate_training_data(test_corpus, word_to_idx, window_size=2)

print(f"Generated {len(test_pairs)} training pairs")
print("\nFirst 10 pairs (center_word, context_word):")
for center_idx, context_idx in test_pairs[:10]:
    center_word = idx_to_word[center_idx]
    context_word = idx_to_word[context_idx]
    print(f" ({center_word}, {context_word})")

Generated 18 training pairs

First 10 pairs (center_word, context_word):
 (the, cat)
 (the, sat)
 (cat, the)
 (cat, sat)
 (cat, on)
 (sat, the)
 (sat, cat)
 (sat, on)
 (sat, the)
 (on, cat)


### Generate pairs for full corpus

Now apply to our entire mini corpus to see total training examples.

In [164]:
all_pairs = generate_training_data(corpus, word_to_idx, window_size=2)

print(f"Total training pairs: {len(all_pairs)}")
print(f"Corpus size: {len(all_words)} words")
print(f"Vocabulary size: {vocab_size} unique words")
print(f"\nAverage pairs per word: {len(all_pairs) / len(all_words):.2f}")

Total training pairs: 146
Corpus size: 47 words
Vocabulary size: 29 unique words

Average pairs per word: 3.11


## 6. Forward Pass Implementation

### Naive softmax version (for understanding)

For one (center, context) pair, compute:
1. Embedding lookup for center word
2. Scores for all vocabulary words
3. Softmax to get probabilities
4. Cross-entropy loss

In [165]:
embed_dim = 50
np.random.seed(42)

W_in = np.random.randn(vocab_size, embed_dim) * 0.01
W_out = np.random.randn(vocab_size, embed_dim) * 0.01

print(f"W_in shape: {W_in.shape}")
print(f"W_out shape: {W_out.shape}")
print(f"Total params: {W_in.shape + W_out.shape}")

W_in shape: (29, 50)
W_out shape: (29, 50)
Total params: (29, 50, 29, 50)


In [166]:
def forward_pass(center_idx, context_idx, W_in, W_out):
    """
    Comutre forward pass for one training pair.

    Args: 
        center_idx: Index of center word
        context_idx: Index of context word (target)
        W_in: Input embedding matrix (vocab_size, embed_dim)
        W_out: Output embedding matrix (vocab_size, embed_dim)

    Returns: 
        loss: Scalar loss value
        cache: Dictionary with intermediate values (for backward pass)
    """

    # Step 1: Get center word embedding
    v_center = W_in[center_idx]

    # Step 2: Compute scores for all words
    scores = v_center @ W_out.T

    # Step 3: Stable softmax
    scores_shifted = scores - np.max(scores)
    exp_scores = np.exp(scores_shifted)
    probs = exp_scores / np.sum(exp_scores)

    # Step 4: Cross-entropy loss
    loss = -np.log(probs[context_idx])

    # Cache values for backward pass

    cache = {
        'center_idx': center_idx,
        'context_idx': context_idx,
        'v_center': v_center,
        'scores': scores,
        'probs': probs
    }

    return loss, cache

In [167]:
# Test forward pass
test_center = word_to_idx["cat"]
test_context = word_to_idx["sat"]

loss, cache = forward_pass(test_center, test_context, W_in, W_out)

print(f"Center word: 'cat' (idx={test_center})")
print(f"Context word: 'sat' (idx={test_context})")
print(f"Loss: {loss:.4f}")
print(f"Probability for 'sat': {cache['probs'][test_context]:.4f}")
print(f"\nTop 5 most probable context words:")
top5_indices = np.argsort(cache['probs'])[-5:][::-1]
for idx in top5_indices:
    print(f"  {idx_to_word[idx]}: {cache['probs'][idx]:.4f}")

Center word: 'cat' (idx=5)
Context word: 'sat' (idx=21)
Loss: 3.3671
Probability for 'sat': 0.0345

Top 5 most probable context words:
  the: 0.0345
  a: 0.0345
  cat: 0.0345
  enemies: 0.0345
  and: 0.0345


## 7. Backward Pass – Gradient Derivation

Goal: Compute gradients of loss w.r.t. W_in and W_out.

### Math refresher:

**Loss function:**
```
L = -log(softmax(scores)[context_idx])
```

**Chain rule:**
```
dL/dW_in = dL/dprobs × dprobs/dscores × dscores/dW_in
```

We need to derive each component step-by-step.
```

In [168]:
def backward_pass(cache, W_in, W_out):
    """
    Compute gradients for one training pair.

    Args:
        cache: Dictionary from forward pass
        W_in, W_out: Embedding matrices

    Return:
        dW_in: Gradient for input embeddings (vocab_size, embed_dim)
        dW_out: Gradient for output embeddings (vocab_size, embed_dim)
    """
    center_idx = cache['center_idx']
    context_idx = cache['context_idx']
    v_center = cache['v_center']
    probs = cache['probs']

    # Step 1: Gradient w.r.t. scores
    dscores = probs.copy()
    dscores[context_idx] -= 1

    # Step 2: Gradient w.r.t. W_out
    dW_out = np.outer(dscores, v_center)

    # Step 3: Gradient w.r.t. v_center
    dv_center = W_out.T @ dscores

    # Step 4: Gradient w.r.t. W_in (only update center word row)
    dW_in = np.zeros_like(W_in)
    dW_in[center_idx] = dv_center

    return dW_in, dW_out

In [169]:
# Test backward pass
dW_in, dW_out = backward_pass(cache, W_in, W_out)

print(f"dW_in shape: {dW_in.shape}")
print(f"dW_out shape: {dW_out.shape}")
print(f"\ndW_in non-zero rows: {np.sum(np.any(dW_in != 0, axis=1))}")  # Should be 1
print(f"dW_in[center_idx] norm: {np.linalg.norm(dW_in[test_center]):.4f}")
print(f"dW_out mean abs gradient: {np.mean(np.abs(dW_out)):.6f}")

dW_in shape: (29, 50)
dW_out shape: (29, 50)

dW_in non-zero rows: 1
dW_in[center_idx] norm: 0.0767
dW_out mean abs gradient: 0.000570


## 8. Numerical Gradient Check

Verify analytical gradients by comparing with finite differences approximation:
```
gradient ≈ (f(x + ε) - f(x - ε)) / (2ε)
```

If analytical and numerical gradients match → our backprop is correct!

In [170]:
def numerical_gradient_check(center_idx, context_idx, W_in, W_out, epsilon=1e-5):
    """
    Compute numerical gradients and compare with analytical gradients.
    
    Returns relative error (should be < 1e-7 for correct implementation).
    """
    # Get analytical gradients
    loss, cache = forward_pass(center_idx, context_idx, W_in, W_out)
    dW_in_analytical, dW_out_analytical = backward_pass(cache, W_in, W_out)
    
    # Check 5 random positions in W_in
    print("Checking W_in gradients (5 random positions)...")
    for _ in range(5):
        i = np.random.randint(0, W_in.shape[0])
        j = np.random.randint(0, W_in.shape[1])
        
        # Numerical gradient for W_in
        W_in_plus = W_in.copy()
        W_in_plus[i, j] += epsilon
        loss_plus, _ = forward_pass(center_idx, context_idx, W_in_plus, W_out)
        
        W_in_minus = W_in.copy()
        W_in_minus[i, j] -= epsilon
        loss_minus, _ = forward_pass(center_idx, context_idx, W_in_minus, W_out)
        numerical_grad = (loss_plus - loss_minus) / (2 * epsilon)
        analytical_grad = dW_in_analytical[i, j]
        
        if abs(numerical_grad) + abs(analytical_grad) > 1e-10:
            rel_error = abs(numerical_grad - analytical_grad) / (abs(numerical_grad) + abs(analytical_grad))
        else:
            rel_error = 0.0
        
        print(f"  W_in[{i},{j}]: numerical={numerical_grad:.8f}, analytical={analytical_grad:.8f}, rel_error={rel_error:.2e}")
    
    # Check 5 random positions in W_out
    print("\nChecking W_out gradients (5 random positions)...")
    for _ in range(5):
        i = np.random.randint(0, W_out.shape[0])
        j = np.random.randint(0, W_out.shape[1])
        
        W_out_plus = W_out.copy()
        W_out_plus[i, j] += epsilon
        loss_plus, _ = forward_pass(center_idx, context_idx, W_in, W_out_plus)
        
        W_out_minus = W_out.copy()
        W_out_minus[i, j] -= epsilon
        loss_minus, _ = forward_pass(center_idx, context_idx, W_in, W_out_minus)
        
        numerical_grad = (loss_plus - loss_minus) / (2 * epsilon)
        analytical_grad = dW_out_analytical[i, j]
        
        if abs(numerical_grad) + abs(analytical_grad) > 1e-10:
            rel_error = abs(numerical_grad - analytical_grad) / (abs(numerical_grad) + abs(analytical_grad))
        else:
            rel_error = 0.0
        
        print(f"  W_out[{i},{j}]: numerical={numerical_grad:.8f}, analytical={analytical_grad:.8f}, rel_error={rel_error:.2e}")
    
    print("\nIf all relative errors < 1e-5, gradients are CORRECT!")


np.random.seed(43)
numerical_gradient_check(test_center, test_context, W_in, W_out)

Checking W_in gradients (5 random positions)...
  W_in[4,0]: numerical=0.00000000, analytical=0.00000000, rel_error=0.00e+00
  W_in[17,21]: numerical=0.00000000, analytical=0.00000000, rel_error=0.00e+00
  W_in[26,16]: numerical=0.00000000, analytical=0.00000000, rel_error=0.00e+00
  W_in[19,17]: numerical=0.00000000, analytical=0.00000000, rel_error=0.00e+00
  W_in[27,27]: numerical=0.00000000, analytical=0.00000000, rel_error=0.00e+00

Checking W_out gradients (5 random positions)...
  W_out[2,46]: numerical=0.00031050, analytical=0.00031050, rel_error=1.49e-10
  W_out[23,0]: numerical=-0.00043486, analytical=-0.00043486, rel_error=1.91e-08
  W_out[3,11]: numerical=-0.00002052, analytical=-0.00002052, rel_error=1.34e-07
  W_out[25,1]: numerical=0.00031704, analytical=0.00031704, rel_error=5.23e-09
  W_out[15,34]: numerical=0.00073463, analytical=0.00073463, rel_error=9.88e-09

If all relative errors < 1e-5, gradients are CORRECT!
